In [1]:
import json
import re
from pathlib import Path
from typing import Dict, Set

# =========================
# CONFIG
# =========================

BASE_DIR = Path(r"C:\Users\cyfij\OneDrive\Desktop\DFRWS 2026\Agent\RQs\normalized_results")

GT_PATH = BASE_DIR / "ground_truth" / "corpus_level" / "corpus_level.jsonl"

METHODS = {
    "Gemini-2.5-Pro": {"path": "gemini_2_5_pro", "bm": "88.4\\%"},
    "GPT-3.5-Turbo": {"path": "gpt_3_5_turbo", "bm": "xx"},
    "GPT-4.1": {"path": "gpt_4_1", "bm": "xx"},
    "GPT-4o-mini": {"path": "gpt_4o_mini", "bm": "82.0\\%"},
    "GPT-5.1": {"path": "gpt_5_1", "bm": "xx"},
    "LLaMA-3.1-8B-Instruct": {"path": "llama_3_1_8b", "bm": "xx"},
    "LLaMA-3.1-70B-Instruct": {"path": "llama_3_1_70b", "bm": "xx"},
    "Mistral-Large": {"path": "mistral_large", "bm": "xx"},
    "Mixtral-8x7B": {"path": "mixtral_8x7b", "bm": "xx"},
    "Mixtral-8x22B": {"path": "mixtral_8x22b", "bm": "xx"},
    "Qwen2.5-72B": {"path": "qwen_2_5_72b", "bm": "82.3\\%"},
}

PII_ORDER = ["EMAIL", "PHONE", "USERNAME", "PERSON_NAME"]

# =========================
# CANONICALIZATION
# =========================

def canonicalize(val: str, pii_type: str) -> str:
    val = val.strip()

    if pii_type == "EMAIL":
        return val.lower()

    if pii_type == "PHONE":
        plus = val.startswith("+")
        digits = re.sub(r"\D", "", val)
        return "+" + digits if plus else digits

    if pii_type in {"USERNAME", "PERSON_NAME"}:
        val = val.lower()
        val = re.sub(r"\b(mr|ms|mrs|dr|prof)\.?\b", "", val)
        val = re.sub(r"[^\w\s]", "", val)
        val = re.sub(r"\s+", " ", val)
        return val.strip()

    return val


# =========================
# LOAD CORPUS
# =========================

def load_corpus(path: Path) -> Dict[str, Set[str]]:
    data = {t: set() for t in PII_ORDER}
    if not path.exists():
        return data

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            pii_type = (rec.get("PII_type") or "").upper()
            if pii_type not in PII_ORDER:
                continue

            vals = rec.get("PII_unique") or rec.get("PII_all") or []
            canon_vals = {
                canonicalize(v, pii_type)
                for v in vals
                if isinstance(v, str) and v.strip()
            }

            data[pii_type].update(canon_vals)

    return data

# =========================
# LOAD GT
# =========================

GT = load_corpus(GT_PATH)

# =========================
# COMPUTE TABLE
# =========================

rows = []

for name, info in METHODS.items():

    corpus_path = BASE_DIR / info["path"] / "corpus_level" / "corpus_level.jsonl"
    SYS = load_corpus(corpus_path)

    total_sys = 0
    total_gt = 0
    total_overlap = 0
    counts = {}

    for t in PII_ORDER:

        gt_set = GT[t]
        sys_set = SYS[t]

        counts[t] = len(sys_set)

        overlap = gt_set.intersection(sys_set)

        total_sys += len(sys_set)
        total_gt += len(gt_set)
        total_overlap += len(overlap)

    precision = (total_overlap / total_sys) if total_sys else 0.0
    recall = (total_overlap / total_gt) if total_gt else 0.0

    rows.append((
        name,
        info["bm"],
        counts["EMAIL"],
        counts["PHONE"],
        counts["USERNAME"],
        counts["PERSON_NAME"],
        f"{precision*100:.2f}\\%",
        f"{recall*100:.2f}\\%",
    ))

# =========================
# EMIT LATEX
# =========================

print(r"\begin{table*}")
print(r"\centering")
print(r"\small")
print(r"\caption{Distribution of distinct PII entities discovered across evaluated methods and PII categories.}")
print(r"\label{tab:model_yield}")
print(r"\begin{tabular}{|l|l|l|l|l|l|l|l|}")
print(r"\hline")
print(r"\multicolumn{2}{|c|}{\textbf{Method/LLM}} & \textbf{Email} & \textbf{Phone} & \textbf{User Name} & \textbf{Real Name} & \textbf{Precision} & \textbf{Recall} \\")
print(r"\hline")

for r in rows:
    print(" & ".join(map(str, r)) + r" \\")
    print(r"\hline")

print(r"\end{tabular}")
print(r"\end{table*}")


\begin{table*}
\centering
\small
\caption{Distribution of distinct PII entities discovered across evaluated methods and PII categories.}
\label{tab:model_yield}
\begin{tabular}{|l|l|l|l|l|l|l|l|}
\hline
\multicolumn{2}{|c|}{\textbf{Method/LLM}} & \textbf{Email} & \textbf{Phone} & \textbf{User Name} & \textbf{Real Name} & \textbf{Precision} & \textbf{Recall} \\
\hline
Gemini-2.5-Pro & 88.4\% & 10 & 791 & 1664 & 1076 & 76.50\% & 49.03\% \\
\hline
GPT-3.5-Turbo & xx & 4 & 13 & 0 & 276 & 24.23\% & 1.29\% \\
\hline
GPT-4.1 & xx & 1289 & 680 & 531 & 683 & 33.30\% & 19.19\% \\
\hline
GPT-4o-mini & 82.0\% & 14 & 22 & 875 & 1928 & 39.98\% & 20.54\% \\
\hline
GPT-5.1 & xx & 16 & 1184 & 1234 & 2154 & 51.20\% & 42.52\% \\
\hline
LLaMA-3.1-8B-Instruct & xx & 0 & 0 & 2 & 15 & 88.24\% & 0.27\% \\
\hline
LLaMA-3.1-70B-Instruct & xx & 6 & 6 & 34 & 15 & 39.34\% & 0.43\% \\
\hline
Mistral-Large & xx & 2 & 989 & 2121 & 1 & 59.56\% & 33.56\% \\
\hline
Mixtral-8x7B & xx & 6 & 6 & 2302 & 98 & 48.34\% & 21.10